# NammaFlow — Resume saved evaluation
Loads the saved model; does not retrain, tune, commit, or push. Run cells in order. Stop on an error.
The model metadata records scikit-learn 1.6.1; investigate any version warning before trusting results.

In [ ]:

# NammaFlow — Restore GitHub Repository

import os
from pathlib import Path

repo_path = Path("/content/NammaFlow")

if not repo_path.exists():
    !git clone https://github.com/Narasimhan15/NammaFlow.git /content/NammaFlow
else:
    print("Repository already exists.")

print("\nRESTORATION CHECK")

paths = {
    "Repository": repo_path,
    "Ward intelligence data": repo_path / "data/features/ward_intelligence.csv",
    "Saved Random Forest": repo_path / "models/nammaflow_ward_conversion_rf_v1.joblib"
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")


In [ ]:

# NammaFlow — Restore Dataset and Saved Model

import pandas as pd
import joblib
from pathlib import Path

repo = Path("/content/NammaFlow")

# Load Phase 3 feature-engineered dataset
ward_ml = pd.read_csv(
    repo / "data/features/ward_intelligence.csv"
)

# Load previously trained Random Forest
rf_model = joblib.load(
    repo / "models/nammaflow_ward_conversion_rf_v1.joblib"
)

# Verify restored objects
print("DATASET SHAPE:", ward_ml.shape)
print("MODEL TYPE:", type(rf_model).__name__)

print("\nMODEL FEATURES:")
features = list(rf_model.feature_names_in_)
print(features)

print("\nFEATURE VALIDATION:")
for feature in features:
    print(f"{feature}: {'FOUND' if feature in ward_ml.columns else 'MISSING'}")

print("\nTARGET VALIDATION:")
target = "ward_search_to_booking_rate"
print(f"{target}: {'FOUND' if target in ward_ml.columns else 'MISSING'}")

print("\nMODEL PARAMETERS:")
print("Trees:", rf_model.n_estimators)
print("Max depth:", rf_model.max_depth)
print("Minimum leaf samples:", rf_model.min_samples_leaf)


In [ ]:

# NammaFlow — Restore Original Train/Test Split

from sklearn.model_selection import train_test_split

target = "ward_search_to_booking_rate"

candidate_features = [
    "ward_searches",
    "active_drvr_auto_cab",
    "supply_vehicle_coverage",
    "has_cab_supply"
]

final_features = list(rf_model.feature_names_in_)

# Reconstruct original modelling datasets
model_df = ward_ml[
    ["ward_num"] + candidate_features + [target]
].copy()

model_final = model_df[
    ["ward_num"] + final_features + [target]
].copy()

# Validate data before splitting
assert model_final["ward_num"].is_unique
assert model_final[final_features + [target]].notna().all().all()

X = model_final[final_features].copy()
y = model_final[target].copy()
ward_ids = model_final["ward_num"].copy()

# Reproduce original 80/20 split
X_train, X_test, y_train, y_test, ward_train, ward_test = (
    train_test_split(
        X, y, ward_ids,
        test_size=0.20,
        random_state=42
    )
)

# Restore saved model predictions
y_pred_rf = rf_model.predict(X_test)

# Verification
print("RESTORED DATASET SHAPES")
print("model_df:", model_df.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nMODEL PREDICTIONS:", len(y_pred_rf))
print("TRAIN/TEST OVERLAP:",
      len(set(X_train.index) & set(X_test.index)))

print("\nRECOVERY SUCCESSFUL")


In [ ]:

# 4.11.7.2 — High-Demand Ward Error Contribution

import numpy as np
import pandas as pd

# Verify test-set alignment
assert X_test.index.equals(y_test.index)
assert X_test.index.equals(ward_test.index)

# Combine actual values, predictions and demand
sensitivity_df = pd.DataFrame({
    "ward_id": ward_test.to_numpy(),
    "ward_searches": X_test["ward_searches"].to_numpy(),
    "actual_rate_pct": y_test.to_numpy() * 100,
    "predicted_rate_pct": rf_model.predict(X_test) * 100
}, index=X_test.index)

# Calculate prediction error in percentage points
sensitivity_df["absolute_error_pp"] = (
    sensitivity_df["actual_rate_pct"]
    - sensitivity_df["predicted_rate_pct"]
).abs()

# Calculate contribution to demand-weighted MAE
total_demand = sensitivity_df["ward_searches"].sum()

sensitivity_df["weighted_error_contribution_pp"] = (
    sensitivity_df["absolute_error_pp"]
    * sensitivity_df["ward_searches"]
    / total_demand
)

weighted_mae_pp = (
    sensitivity_df["weighted_error_contribution_pp"].sum()
)

# Percentage contribution to total weighted error
sensitivity_df["error_contribution_pct"] = (
    sensitivity_df["weighted_error_contribution_pp"]
    / weighted_mae_pp * 100
)

# Display highest-demand wards
print("HIGH-DEMAND WARD ERROR ANALYSIS")

display(
    sensitivity_df.sort_values(
        "ward_searches", ascending=False
    ).head(10).round(2)
)

print("\nDEMAND-WEIGHTED MODEL PERFORMANCE")
print(
    "Demand-weighted MAE:",
    round(weighted_mae_pp, 4),
    "percentage points"
)

print("\nHIGHEST ERROR CONTRIBUTORS")
display(
    sensitivity_df.sort_values(
        "weighted_error_contribution_pp",
        ascending=False
    ).head(5).round(2)
)


In [ ]:

# 4.11.8.1 — Demand-Segment Performance Analysis

import pandas as pd
import numpy as np

segment_df = sensitivity_df.copy()

# Create three similarly sized groups by demand rank
segment_df["demand_segment"] = pd.qcut(
    segment_df["ward_searches"].rank(method="first"),
    q=3,
    labels=["Low Demand", "Medium Demand", "High Demand"]
)

# Training-mean baseline prediction
baseline_pct = y_train.mean() * 100

segment_df["baseline_abs_error_pp"] = (
    segment_df["actual_rate_pct"] - baseline_pct
).abs()

results = []
total_demand = segment_df["ward_searches"].sum()

for segment in [
    "Low Demand",
    "Medium Demand",
    "High Demand"
]:
    subset = segment_df[
        segment_df["demand_segment"] == segment
    ]

    weights = subset["ward_searches"]

    rf_mae = np.average(
        subset["absolute_error_pp"],
        weights=weights
    )

    baseline_mae = np.average(
        subset["baseline_abs_error_pp"],
        weights=weights
    )

    improvement = (
        (baseline_mae - rf_mae) / baseline_mae * 100
        if baseline_mae > 0 else np.nan
    )

    results.append({
        "demand_segment": segment,
        "ward_count": len(subset),
        "total_searches": weights.sum(),
        "demand_share_pct": weights.sum() / total_demand * 100,
        "rf_mae_pp": rf_mae,
        "baseline_mae_pp": baseline_mae,
        "improvement_pct": improvement
    })

segment_results = pd.DataFrame(results)

print("NAMMAFLOW — DEMAND-SEGMENT PERFORMANCE")
display(segment_results.round(4))

print("\nVALIDATION")
print("Total wards:", segment_results["ward_count"].sum())
print("Total searches:", segment_results["total_searches"].sum())
print(
    "Demand share total:",
    round(segment_results["demand_share_pct"].sum(), 2),
    "%"
)


In [ ]:

# 4.11.8.2 — Demand-Segment Performance Visualization

import matplotlib.pyplot as plt
import numpy as np

plot_df = segment_results.copy()

x = np.arange(len(plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))

# Random Forest MAE
bars_rf = ax.bar(
    x - width / 2,
    plot_df["rf_mae_pp"],
    width,
    label="Random Forest"
)

# Baseline MAE
bars_base = ax.bar(
    x + width / 2,
    plot_df["baseline_mae_pp"],
    width,
    label="Mean Baseline"
)

# Add values above bars
ax.bar_label(bars_rf, fmt="%.2f", padding=3)
ax.bar_label(bars_base, fmt="%.2f", padding=3)

# Include demand share in category labels
labels = [
    f"{row['demand_segment']}\n"
    f"({row['demand_share_pct']:.2f}% of searches)"
    for _, row in plot_df.iterrows()
]

ax.set_xticks(x)
ax.set_xticklabels(labels)

ax.set_title(
    "NammaFlow — Model Performance by Demand Segment",
    fontsize=14,
    pad=18
)

ax.set_xlabel("Ward Demand Segment")
ax.set_ylabel("Demand-Weighted MAE (Percentage Points)")

ax.set_ylim(
    0,
    max(
        plot_df["rf_mae_pp"].max(),
        plot_df["baseline_mae_pp"].max()
    ) * 1.2
)

ax.legend()
ax.grid(axis="y", alpha=0.2)

fig.tight_layout()
plt.show()


In [ ]:

# 4.11.8.3 — Ward-Level Error Distribution

import pandas as pd
import numpy as np

diagnostic_results = []

for segment in [
    "Low Demand",
    "Medium Demand",
    "High Demand"
]:
    subset = segment_df[
        segment_df["demand_segment"] == segment
    ].copy()

    # Compare ward-level prediction accuracy
    rf_better = (
        subset["absolute_error_pp"]
        < subset["baseline_abs_error_pp"]
    )

    diagnostic_results.append({
        "segment": segment,
        "ward_count": len(subset),
        "median_error_pp": subset[
            "absolute_error_pp"
        ].median(),
        "max_error_pp": subset[
            "absolute_error_pp"
        ].max(),
        "wards_error_gt_10pp": (
            subset["absolute_error_pp"] > 10
        ).sum(),
        "rf_better_wards_pct": (
            rf_better.mean() * 100
        )
    })

diagnostic_df = pd.DataFrame(diagnostic_results)

print("WARD-LEVEL ERROR DIAGNOSTICS")
display(diagnostic_df.round(2))

print("\nVALIDATION")
print("Total wards:", diagnostic_df["ward_count"].sum())

print(
    "Wards with error > 10 pp:",
    diagnostic_df["wards_error_gt_10pp"].sum()
)

print(
    "Maximum observed error:",
    round(segment_df["absolute_error_pp"].max(), 2),
    "percentage points"
)


In [ ]:
assert len(segment_df) == 49
assert int(diagnostic_df["wards_error_gt_10pp"].sum()) == 16
assert round(float(segment_df["absolute_error_pp"].max()), 2) == 30.55
print("Previous diagnostics reproduced successfully.")